# IPCC AR6 Interactive Atlas — CMIP6 Model Uncertainty

Explores CMIP6 multi-model spread across climate variables for **Western North America** under **SSP3-7.0**, using percentile data extracted from the [IPCC WGI Interactive Atlas](https://interactive-atlas.ipcc.ch/regional-information).

Projected changes are expressed relative to the **1850–1900 baseline** and shown for three future time periods using box-whisker style plots. Three line widths encode different percentile ranges (P5–P95, P10–P90, P25–P75) with the median marked as a dot.

---

## 1. Imports

In [ ]:
import pandas as pd
import plotly.graph_objects as go

## 2. Data

Raw percentile statistics (P5, P10, P25, median, P75, P90, P95) were manually extracted from the [IPCC WGI Interactive Atlas: Regional information](https://interactive-atlas.ipcc.ch/regional-information) for five climate variables across three future time periods (**Near Term 2021–2040**, **Medium Term 2041–2060**, **Long Term 2081–2100**) under **SSP3-7.0** for **Western North America**.

In [ ]:
# Raw data from IPCC AR6 Interactive Atlas — Western North America, SSP3-7.0
data = {
    "Mean temperature (°C)": {
        "Near Term (2021–2040)":   {"median": 1.9, "p25": 1.6, "p75": 2.2, "p10": 1.3, "p90": 2.4, "p5": 1.1,  "p95": 2.7},
        "Medium Term (2041–2060)": {"median": 2.7, "p25": 2.4, "p75": 3.3, "p10": 2.1, "p90": 3.6, "p5": 1.9,  "p95": 4.0},
        "Long Term (2081–2100)":   {"median": 4.8, "p25": 4.2, "p75": 5.4, "p10": 3.8, "p90": 6.1, "p5": None, "p95": None},
    },
    "Max temperature (°C)": {
        "Near Term (2021–2040)":   {"median": 1.9, "p25": 1.6, "p75": 2.1, "p10": 1.6, "p90": 2.3, "p5": 1.2,  "p95": 2.7},
        "Medium Term (2041–2060)": {"median": 2.7, "p25": 2.4, "p75": 3.0, "p10": 2.2, "p90": 3.7, "p5": 2.0,  "p95": 4.0},
        "Long Term (2081–2100)":   {"median": 4.8, "p25": 4.2, "p75": 5.5, "p10": 4.1, "p90": 6.4, "p5": 3.8,  "p95": 6.6},
    },
    "Total precipitation (%)": {
        "Near Term (2021–2040)":   {"median": 1.9,  "p25": -0.9, "p75": 3.7, "p10": -2.1, "p90": 5.8,  "p5": -2.7, "p95": 8.9},
        "Medium Term (2041–2060)": {"median": 3.3,  "p25":  1.4, "p75": 4.6, "p10": -0.1, "p90": 8.8,  "p5": -1.1, "p95": 10.6},
        "Long Term (2081–2100)":   {"median": 7.7,  "p25":  3.7, "p75": 9.6, "p10":  2.1, "p90": 13.3, "p5": -0.7, "p95": 13.8},
    },
    "Surface wind (%)": {
        "Near Term (2021–2040)":   {"median": -1.0, "p25": -1.7, "p75": -0.0, "p10": -2.6, "p90": 1.2,  "p5": -4.0, "p95": 2.7},
        "Medium Term (2041–2060)": {"median": -1.6, "p25": -2.2, "p75": -0.8, "p10": -4.2, "p90": 0.3,  "p5": -4.4, "p95": 2.2},
        "Long Term (2081–2100)":   {"median": -3.2, "p25": -4.6, "p75": -2.3, "p10": -8.4, "p90": -1.1, "p5": -8.9, "p95": 1.0},
    },
    "SST (°C)": {
        "Near Term (2021–2040)":   {"median": 1.2, "p25": 0.8, "p75": 1.4, "p10": 0.4, "p90": 1.6, "p5": 0.3, "p95": 1.7},
        "Medium Term (2041–2060)": {"median": 1.8, "p25": 1.4, "p75": 2.0, "p10": 1.1, "p90": 2.4, "p5": 1.0, "p95": 2.8},
        "Long Term (2081–2100)":   {"median": 3.1, "p25": 2.7, "p75": 3.7, "p10": 2.3, "p90": 4.2, "p5": 2.2, "p95": 5.0},
    },
}

In [ ]:
# Build dataframe
rows = []
for var, periods in data.items():
    for period, vals in periods.items():
        rows.append({"variable": var, "period": period, **vals})

df = pd.DataFrame(rows)

# derived spread metrics
df["iqr"] = df["p75"] - df["p25"]          # interquartile range
df["range_80"] = df["p90"] - df["p10"]          # 10th–90th range
df["cv"] = df["iqr"] / df["median"].abs()  # coefficient of variation

print(df[["variable", "period", "median", "p25", "p75",
          "iqr", "range_80", "cv"]].to_string(index=False))

## 3. Visualization

Box-whisker plot of CMIP6 model spread per variable and time period. Each variable shows three jittered rows — one per period — using color to distinguish them. The chart is exported as both interactive HTML and a high-resolution PNG.

In [ ]:
# Interactive plot
periods = ["Near Term (2021–2040)",
           "Medium Term (2041–2060)", "Long Term (2081–2100)"]
variables = list(data.keys())

COLORS = {
    "Near Term (2021–2040)":   "#4472A8",
    "Medium Term (2041–2060)": "#C0664A",
    "Long Term (2081–2100)":   "#5A9E6F",
}

PERIOD_SHORT = {
    "Near Term (2021–2040)":   "Near Term",
    "Medium Term (2041–2060)": "Medium Term",
    "Long Term (2081–2100)":   "Long Term",
}

fig = go.Figure()

for vi, var in enumerate(variables):
    for pi, period in enumerate(periods):
        row = df[(df["variable"] == var) & (df["period"] == period)].iloc[0]
        color = COLORS[period]
        offset = (pi - 1) * 0.22   # jitter periods within each variable

        # p5–p95 whisker (thin)
        if row["p5"] is not None and row["p95"] is not None:
            fig.add_trace(go.Scatter(
                x=[row["p5"], row["p95"]],
                y=[vi + offset, vi + offset],
                mode="lines",
                line=dict(color=color, width=1.5),
                showlegend=False,
            ))

        # p10–p90 bar (medium)
        fig.add_trace(go.Scatter(
            x=[row["p10"], row["p90"]],
            y=[vi + offset, vi + offset],
            mode="lines",
            line=dict(color=color, width=4),
            showlegend=False,
        ))

        # p25–p75 bar (thick)
        fig.add_trace(go.Scatter(
            x=[row["p25"], row["p75"]],
            y=[vi + offset, vi + offset],
            mode="lines",
            line=dict(color=color, width=9),
            showlegend=False,
        ))

        # median dot
        fig.add_trace(go.Scatter(
            x=[row["median"]],
            y=[vi + offset],
            mode="markers",
            marker=dict(color="white", size=8,
                        line=dict(color=color, width=2)),
            name=PERIOD_SHORT[period],
            legendgroup=period,
            showlegend=(vi == 0),   # only show legend once per period
        ))

        # zero reference line
        fig.add_vline(x=0, line=dict(color="grey", width=0.8, dash="dash"))

fig.update_layout(
    title=dict(
        text=(
            "CMIP6 inter-model spread by variable, Western North America (SSP3-7.0)<br>"
            "<sup>Whiskers show 5th–95th percentile range, not full min–max</sup>"
        ),
        font=dict(size=13),
        x=0.5,
        xanchor="center",
    ),
    xaxis_title="Projected change relative to 1850–1900",
    height=480,
    width=820,
    plot_bgcolor="white",
    margin=dict(l=200, r=30, t=100, b=60),
    legend=dict(
        orientation="h",
        y=-0.15,
        x=0.5,
        xanchor="center",
        font=dict(size=11),
    ),
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(len(variables))),
        ticktext=variables,
        showgrid=False,
        zeroline=False,
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor="#eeeeee",
        zeroline=False,
    ),
)


fig.show()
fig.write_html("../uncertainty/figures/html/ipcc_atlas_uncertainty.html")
fig.write_image(
    "../uncertainty/figures/static/ipcc_atlas_uncertainty.png", scale=3)